In [15]:
import pandas as pd
import numpy as np

In [16]:
claims_1 = pd.read_csv('claims_staged_cmu_1_2.csv')
claims_2 = pd.read_csv('claims_staged_cmu_2_2.csv')

# merge claims
claims = pd.concat([claims_1, claims_2], ignore_index=True)

In [17]:
claims

,PRI_DIAG_CD,ADM_DIAG_CD,DIAG_CD_001,DIAG_CD_002,DIAG_CD_003,HCPCS,DRG,Med_Cost_Category,NDC_CD,DRUG_NM,...,provider_number,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,PROC_DESC,DIAG_DESC,DRG_DESC,ADM_DIAG_DESC,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC
0,J431,NaN,NaN,NaN,NaN,94727,NaN,Prof PCP Procedures,NaN,NaN,...,NaN,NaN,NaN,Pulm function test by gas,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
1,J431,NaN,NaN,NaN,NaN,94060,NaN,Prof PCP Procedures,NaN,NaN,...,NaN,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
2,J431,NaN,NaN,NaN,NaN,94060,NaN,OP Other,NaN,NaN,...,2236.0,NaN,NaN,Evaluation of wheezing,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
3,J431,NaN,NaN,NaN,NaN,94726,NaN,OP Other,NaN,NaN,...,2236.0,NaN,NaN,Pulm funct tst plethysmograp,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
4,J431,NaN,NaN,NaN,NaN,94729,NaN,OP Other,NaN,NaN,...,2236.0,NaN,NaN,Co/membane diffuse capacity,Panlobular emphysema,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2517382,C50411,NaN,Z170,E1165,Z794,83036,NaN,OP Path/Lab,NaN,NaN,...,8094.0,NaN,NaN,Glycosylated hemoglobin test,Malig neoplm of upper-outer quadrant of right ...,NaN,NaN,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin
2517383,C50411,NaN,Z170,E1165,Z794,82607,NaN,OP Path/Lab,NaN,NaN,...,8094.0,NaN,NaN,Vitamin B-12,Malig neoplm of upper-outer quadrant of right ...,NaN,NaN,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin
2517384,C50411,NaN,Z170,E1165,Z794,80061,NaN,OP Path/Lab,NaN,NaN,...,8094.0,NaN,NaN,Lipid panel,Malig neoplm of upper-outer quadrant of right ...,NaN,NaN,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin
2517385,C50411,NaN,Z170,E1165,Z794,85025,NaN,OP Path/Lab,NaN,NaN,...,8094.0,NaN,NaN,Complete cbc w/auto diff wbc,Malig neoplm of upper-outer quadrant of right ...,NaN,NaN,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin


In [18]:
# drop sparse columns
claims.drop(
    columns=[
        'DRG_DESC',
        'DRG',
        'ADM_DIAG_DESC',
        'ADM_DIAG_CD'
    ],
    inplace=True
)

In [19]:
# type check
claims['days_since_earliest_dt'] = claims['days_since_earliest_dt'].astype(int)
claims['c_allowed'] = claims['c_allowed'].astype(float)

In [20]:
# clean primary diagnosis
claims['PRI_DIAG_CD'] = (
    claims['PRI_DIAG_CD']
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace('.', '', regex=False)
)

claims.loc[claims['PRI_DIAG_CD'] == 'NAN', 'PRI_DIAG_CD'] = np.nan

claims['ICD10_simple'] = claims['PRI_DIAG_CD'].str[:3]

In [21]:
# cleaning of procedure fields
claims['HCPCS'] = (
    claims['HCPCS']
        .astype(str)
        .str.strip()
        .str.upper()
)

claims.loc[claims['HCPCS'] == 'NAN', 'HCPCS'] = np.nan

claims['PROC_DESC'] = (
    claims['PROC_DESC']
        .astype(str)
        .str.strip()
        .str.upper()
)

claims.loc[claims['PROC_DESC'] == 'NAN', 'PROC_DESC'] = np.nan

In [22]:
# clean secondary diagnoses

diag_cols = ['DIAG_CD_001', 'DIAG_CD_002', 'DIAG_CD_003']

for col in diag_cols:
    claims[col] = (
        claims[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.replace('.', '', regex=False)
    )
    claims.loc[claims[col] == 'NAN', col] = np.nan

In [23]:
# standardize drug fields

claims['DRUG_NM'] = claims['DRUG_NM'].astype(str).str.strip().str.upper()
claims.loc[claims['DRUG_NM'] == 'NAN', 'DRUG_NM'] = np.nan

In [25]:
# flag cancer-related claims based on diagnosis codes
def is_cancer(row):
    diag_fields = [
        row['PRI_DIAG_CD'],
        row['DIAG_CD_001'],
        row['DIAG_CD_002'],
        row['DIAG_CD_003']
    ]
    
    for code in diag_fields:
        if isinstance(code, str) and code.startswith('C'):
            return True
    return False

claims['is_cancer_related'] = claims.apply(is_cancer, axis=1)

In [26]:
claims

,PRI_DIAG_CD,DIAG_CD_001,DIAG_CD_002,DIAG_CD_003,HCPCS,Med_Cost_Category,NDC_CD,DRUG_NM,C_UTIL_CT,c_allowed,...,provider_number,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,PROC_DESC,DIAG_DESC,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC,ICD10_simple,is_cancer_related
0,J431,NaN,NaN,NaN,94727,Prof PCP Procedures,NaN,NaN,1.0,18.500000,...,NaN,NaN,NaN,PULM FUNCTION TEST BY GAS,Panlobular emphysema,NaN,NaN,NaN,J43,False
1,J431,NaN,NaN,NaN,94060,Prof PCP Procedures,NaN,NaN,1.0,20.000000,...,NaN,NaN,NaN,EVALUATION OF WHEEZING,Panlobular emphysema,NaN,NaN,NaN,J43,False
2,J431,NaN,NaN,NaN,94060,OP Other,NaN,NaN,1.0,458.120000,...,2236.0,NaN,NaN,EVALUATION OF WHEEZING,Panlobular emphysema,NaN,NaN,NaN,J43,False
3,J431,NaN,NaN,NaN,94726,OP Other,NaN,NaN,0.0,0.000000,...,2236.0,NaN,NaN,PULM FUNCT TST PLETHYSMOGRAP,Panlobular emphysema,NaN,NaN,NaN,J43,False
4,J431,NaN,NaN,NaN,94729,OP Other,NaN,NaN,0.0,0.000000,...,2236.0,NaN,NaN,CO/MEMBANE DIFFUSE CAPACITY,Panlobular emphysema,NaN,NaN,NaN,J43,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2517382,C50411,Z170,E1165,Z794,83036,OP Path/Lab,NaN,NaN,0.0,17.510413,...,8094.0,NaN,NaN,GLYCOSYLATED HEMOGLOBIN TEST,Malig neoplm of upper-outer quadrant of right ...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,C50,True
2517383,C50411,Z170,E1165,Z794,82607,OP Path/Lab,NaN,NaN,0.0,27.194525,...,8094.0,NaN,NaN,VITAMIN B-12,Malig neoplm of upper-outer quadrant of right ...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,C50,True
2517384,C50411,Z170,E1165,Z794,80061,OP Path/Lab,NaN,NaN,0.0,24.140880,...,8094.0,NaN,NaN,LIPID PANEL,Malig neoplm of upper-outer quadrant of right ...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,C50,True
2517385,C50411,Z170,E1165,Z794,85025,OP Path/Lab,NaN,NaN,0.0,14.008330,...,8094.0,NaN,NaN,COMPLETE CBC W/AUTO DIFF WBC,Malig neoplm of upper-outer quadrant of right ...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,C50,True


In [27]:
claims.columns

Index(['PRI_DIAG_CD', 'DIAG_CD_001', 'DIAG_CD_002', 'DIAG_CD_003', 'HCPCS',
       'Med_Cost_Category', 'NDC_CD', 'DRUG_NM', 'C_UTIL_CT', 'c_allowed',
       'days_since_earliest_dt', 'member_number', 'provider_number',
       'ROUTE DESCRIPTION', 'STRENGTH DESCRIPTION', 'PROC_DESC', 'DIAG_DESC',
       'DIAG_001_DIAG_DESC', 'DIAG_002_DIAG_DESC', 'DIAG_003_DIAG_DESC',
       'ICD10_simple', 'is_cancer_related'],
      dtype='str')

In [28]:
claims = claims[
    [
        # Core identifiers + time
        'member_number',
        'days_since_earliest_dt',

        # Cost + utilization
        'c_allowed',
        'C_UTIL_CT',
        'Med_Cost_Category',
        'is_cancer_related',

        # Primary diagnosis
        'PRI_DIAG_CD',
        'ICD10_simple',
        'DIAG_DESC',

        # Secondary diagnoses
        'DIAG_CD_001',
        'DIAG_CD_002',
        'DIAG_CD_003',
        'DIAG_001_DIAG_DESC',
        'DIAG_002_DIAG_DESC',
        'DIAG_003_DIAG_DESC',

        # Procedures
        'HCPCS',
        'PROC_DESC',

        # Drug information
        'NDC_CD',
        'DRUG_NM',
        'ROUTE DESCRIPTION',
        'STRENGTH DESCRIPTION',

        # Provider
        'provider_number'
    ]
]

claims

,member_number,days_since_earliest_dt,c_allowed,C_UTIL_CT,Med_Cost_Category,is_cancer_related,PRI_DIAG_CD,ICD10_simple,DIAG_DESC,DIAG_CD_001,...,DIAG_001_DIAG_DESC,DIAG_002_DIAG_DESC,DIAG_003_DIAG_DESC,HCPCS,PROC_DESC,NDC_CD,DRUG_NM,ROUTE DESCRIPTION,STRENGTH DESCRIPTION,provider_number
0,A001,-364,18.500000,1.0,Prof PCP Procedures,False,J431,J43,Panlobular emphysema,NaN,...,NaN,NaN,NaN,94727,PULM FUNCTION TEST BY GAS,NaN,NaN,NaN,NaN,NaN
1,A001,-364,20.000000,1.0,Prof PCP Procedures,False,J431,J43,Panlobular emphysema,NaN,...,NaN,NaN,NaN,94060,EVALUATION OF WHEEZING,NaN,NaN,NaN,NaN,NaN
2,A001,-364,458.120000,1.0,OP Other,False,J431,J43,Panlobular emphysema,NaN,...,NaN,NaN,NaN,94060,EVALUATION OF WHEEZING,NaN,NaN,NaN,NaN,2236.0
3,A001,-364,0.000000,0.0,OP Other,False,J431,J43,Panlobular emphysema,NaN,...,NaN,NaN,NaN,94726,PULM FUNCT TST PLETHYSMOGRAP,NaN,NaN,NaN,NaN,2236.0
4,A001,-364,0.000000,0.0,OP Other,False,J431,J43,Panlobular emphysema,NaN,...,NaN,NaN,NaN,94729,CO/MEMBANE DIFFUSE CAPACITY,NaN,NaN,NaN,NaN,2236.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2517382,D660,65,17.510413,0.0,OP Path/Lab,True,C50411,C50,Malig neoplm of upper-outer quadrant of right ...,Z170,...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,83036,GLYCOSYLATED HEMOGLOBIN TEST,NaN,NaN,NaN,NaN,8094.0
2517383,D660,65,27.194525,0.0,OP Path/Lab,True,C50411,C50,Malig neoplm of upper-outer quadrant of right ...,Z170,...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,82607,VITAMIN B-12,NaN,NaN,NaN,NaN,8094.0
2517384,D660,65,24.140880,0.0,OP Path/Lab,True,C50411,C50,Malig neoplm of upper-outer quadrant of right ...,Z170,...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,80061,LIPID PANEL,NaN,NaN,NaN,NaN,8094.0
2517385,D660,65,14.008330,0.0,OP Path/Lab,True,C50411,C50,Malig neoplm of upper-outer quadrant of right ...,Z170,...,Estrogen receptor positive status [ER+],Type 2 diabetes mellitus with hyperglycemia,Long term (current) use of insulin,85025,COMPLETE CBC W/AUTO DIFF WBC,NaN,NaN,NaN,NaN,8094.0


In [29]:
claims.to_csv('cleaned_claims.csv', index=False)